In [3]:
import pandas as pd
from sqlalchemy import create_engine
import os

# SQLite — creates a database FILE on your PC
# No password, no server, no setup needed!
engine = create_engine(r"sqlite:///C:\Users\vikra\Downloads\HR Analytics Project\HR_analytics.db")
print("✅ Connected to SQLite Database!")

✅ Connected to SQLite Database!


In [3]:
file = {
    "HR_analytics":        r"C:\Users\vikra\Downloads\HR Analytics Project\cleaned\HR_Analytics_CLEAN.csv",
    
}

for table_name, file_path in file.items():
    try:
        df = pd.read_csv(file_path, low_memory=False, encoding='latin1')
        df.to_sql(table_name,
                  con=engine,
                  if_exists='replace',
                  index=False,
                  chunksize=1000)
        print(f"✅ Loaded {table_name} → {df.shape[0]} rows")
    except Exception as e:
        print(f"❌ Failed {table_name}: {e}")

print("\n🎉 All tables loaded!")

✅ Loaded HR_analytics → 1240 rows

🎉 All tables loaded!


In [ ]:
#Business Queries

In [5]:
#1. Who is leaving?

#Overall attrition rate
query = """
SELECT 
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS total_left,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM HR_analytics;
"""
result = pd.read_sql(query, engine)
print(result)

   total_employees  total_left  attrition_rate_pct
0             1240         196                15.8


In [8]:
#1. Who is leaving?

#-- Attrition rate by Department
query = """
SELECT 
    Department,
    COUNT(*) AS headcount,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS left_count,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM HR_analytics
GROUP BY Department
ORDER BY attrition_rate_pct DESC;
"""
result = pd.read_sql(query, engine)
print(result)

               Department  headcount  left_count  attrition_rate_pct
0         Human Resources         50          11                22.0
1                   Sales        356          72                20.2
2  Research & Development        796         111                13.9
3                    None         38           2                 5.3


In [9]:
#-- Attrition rate by Jobrole
query = """
SELECT 
    JobRole,
    COUNT(*) AS headcount,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS left_count,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM HR_analytics
GROUP BY JobRole
ORDER BY attrition_rate_pct DESC;
"""
result = pd.read_sql(query, engine)
print(result)

                     JobRole  headcount  left_count  attrition_rate_pct
0       Sales Representative         69          27                39.1
1      Laboratory Technician        218          52                23.9
2            Human Resources         47          11                23.4
3         Research Scientist        255          44                17.3
4            Sales Executive        266          43                16.2
5     Manufacturing Director        123           7                 5.7
6  Healthcare Representative        110           6                 5.5
7                    Manager         85           4                 4.7
8          Research Director         67           2                 3.0


In [10]:
#-- Attrition rate by Agegroup
query = """
SELECT 
    AgeGroup,
    COUNT(*) AS headcount,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS left_count,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM HR_analytics
GROUP BY AgeGroup
ORDER BY attrition_rate_pct DESC;
"""
result = pd.read_sql(query, engine)
print(result)

  AgeGroup  headcount  left_count  attrition_rate_pct
0    18-25        104          38                36.5
1      55+         37           7                18.9
2    26-35        514          94                18.3
3    46-55        183          21                11.5
4    36-45        402          36                 9.0


In [11]:
#2. Why are they leaving?
#-- Avg satisfaction/income for leavers vs stayers
query = """
SELECT 
    Attrition,
    ROUND(AVG(JobSatisfaction), 2) AS avg_job_satisfaction,
    ROUND(AVG(EnvironmentSatisfaction), 2) AS avg_env_satisfaction,
    ROUND(AVG(WorkLifeBalance), 2) AS avg_worklife_balance,
    ROUND(AVG(MonthlyIncome), 0) AS avg_income,
    ROUND(AVG(YearsSinceLastPromotion), 1) AS avg_years_since_promo
FROM HR_analytics
GROUP BY Attrition;
"""
      
result = pd.read_sql(query, engine)
print(result)

  Attrition  avg_job_satisfaction  avg_env_satisfaction  avg_worklife_balance  \
0        No                  2.81                  2.79                  2.77   
1       Yes                  2.47                  2.45                  2.68   

   avg_income  avg_years_since_promo  
0      6797.0                    2.2  
1      4592.0                    1.7  


In [13]:
#-- OverTime vs Attrition
query = """
SELECT 
    OverTime,
    COUNT(*) AS headcount,
    ROUND(100.0 * SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM HR_analytics
GROUP BY OverTime;
"""
result = pd.read_sql(query, engine)
print(result)

  OverTime  headcount  attrition_rate_pct
0       No        890                10.1
1      Yes        350                30.3


In [14]:
#-- Income bracket vs Attrition
query = """
SELECT 
    CASE 
        WHEN MonthlyIncome < 3000 THEN 'Under 3000'
        WHEN MonthlyIncome BETWEEN 3000 AND 6000 THEN '3000-6000'
        WHEN MonthlyIncome BETWEEN 6001 AND 10000 THEN '6001-10000'
        ELSE 'Above 10000'
    END AS income_bracket,
    COUNT(*) AS headcount,
    ROUND(100.0 * SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM HR_analytics
GROUP BY income_bracket
ORDER BY attrition_rate_pct DESC;
"""
result = pd.read_sql(query, engine)
print(result)

  income_bracket  headcount  attrition_rate_pct
0     Under 3000        338                29.0
1      3000-6000        438                12.8
2     6001-10000        235                11.5
3    Above 10000        229                 6.6


In [17]:
#-- Low satisfaction + overtime combo (compounding risk)
query = """
SELECT 
    JobSatisfaction,
    OverTime,
    COUNT(*) AS headcount,
    ROUND(100.0 * SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM HR_analytics
GROUP BY JobSatisfaction,OverTime
ORDER BY attrition_rate_pct DESC;
"""
result = pd.read_sql(query, engine)
print(result)

   JobSatisfaction OverTime  headcount  attrition_rate_pct
0                1      Yes         72                37.5
1                3      Yes        100                35.0
2                2      Yes         51                33.3
3                4      Yes        127                21.3
4                1       No        170                17.6
5                3       No        268                10.1
6                2       No        174                 9.2
7                4       No        278                 6.1


In [19]:
#-- Low salaryhike % vs atrition
query = """
SELECT
    CASE
        WHEN PercentSalaryHike <= 12 THEN 'Low Hike'
        WHEN PercentSalaryHike <= 18 THEN 'Medium Hike'
        ELSE 'High Hike'
    END AS hike_band,
    COUNT(*) AS headcount,
    ROUND(
        100.0 * SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END)
        / COUNT(*),1
    ) AS attrition_rate_pct
FROM HR_analytics
GROUP BY hike_band;
"""
result = pd.read_sql(query, engine)
print(result)

     hike_band  headcount  attrition_rate_pct
0    High Hike        247                15.8
1     Low Hike        350                16.9
2  Medium Hike        643                15.2


In [20]:
#3. What's it costing us?

In [21]:
#-- Estimated annual replacement cost by department
query = """
SELECT 
    Department,
    COUNT(*) AS headcount,
    SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(AVG(MonthlyIncome), 0) AS avg_monthly_income,
    ROUND(SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) * AVG(MonthlyIncome) * 12 * 0.5, 0) AS est_annual_replacement_cost
FROM HR_analytics
GROUP BY Department
ORDER BY est_annual_replacement_cost DESC;
"""
result = pd.read_sql(query, engine)
print(result)

               Department  headcount  attrition_count  avg_monthly_income  \
0  Research & Development        796              111              6242.0   
1                   Sales        356               72              6919.0   
2         Human Resources         50               11              6223.0   
3                    None         38                2              6665.0   

   est_annual_replacement_cost  
0                    4157222.0  
1                    2989145.0  
2                     410688.0  
3                      79983.0  


In [22]:
#-- Total company-wide cost
query = """
SELECT 
    ROUND(SUM(CASE WHEN Attrition='Yes' THEN MonthlyIncome * 12 * 0.5 ELSE 0 END), 0) AS total_estimated_attrition_cost
FROM HR_analytics;
"""
result = pd.read_sql(query, engine)
print(result)

   total_estimated_attrition_cost
0                       5400030.0


In [23]:
#-- Cost by JobRole
query = """
SELECT 
    JobRole,
    SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(AVG(MonthlyIncome) * 12 * 0.5, 0) AS replacement_cost_per_person,
    ROUND(SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) * AVG(MonthlyIncome) * 12 * 0.5, 0) AS total_role_cost
FROM HR_analytics
GROUP BY JobRole
ORDER BY total_role_cost DESC;
"""
result = pd.read_sql(query, engine)
print(result)

                     JobRole  attrition_count  replacement_cost_per_person  \
0            Sales Executive               43                      41398.0   
1      Laboratory Technician               52                      19446.0   
2         Research Scientist               44                      19544.0   
3       Sales Representative               27                      15885.0   
4                    Manager                4                     103431.0   
5     Manufacturing Director                7                      43158.0   
6  Healthcare Representative                6                      45815.0   
7            Human Resources               11                      24953.0   
8          Research Director                2                      94549.0   

   total_role_cost  
0        1780105.0  
1        1011213.0  
2         859930.0  
3         428899.0  
4         413724.0  
5         302109.0  
6         274891.0  
7         274479.0  
8         189099.0  


In [4]:
#4. What should we do? (Diagnostic / actionable)
#-- Departments ranked by raw attrition count (not just rate)
query = """
SELECT 
    Department,
    COUNT(*) AS headcount,
    ROUND(100.0 * SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct,
    SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) AS attrition_count
FROM HR_analytics
GROUP BY Department
ORDER BY attrition_count DESC;
"""
result = pd.read_sql(query, engine)
print(result)

               Department  headcount  attrition_rate_pct  attrition_count
0  Research & Development        796                13.9              111
1                   Sales        356                20.2               72
2         Human Resources         50                22.0               11
3                    None         38                 5.3                2


In [5]:
#- Flight-risk list: still employed, but showing warning signs
query = """
SELECT EmpID, Department, JobRole, JobSatisfaction, OverTime, MonthlyIncome, YearsSinceLastPromotion
FROM HR_analytics
WHERE Attrition = 'No'
  AND JobSatisfaction <= 2
  AND OverTime = 'Yes'
  AND YearsSinceLastPromotion >= 3
ORDER BY JobSatisfaction ASC;
"""
result = pd.read_sql(query, engine)
print(result)

     EmpID              Department                    JobRole  \
0    Rm578  Research & Development         Research Scientist   
1    Rm285  Research & Development  Healthcare Representative   
2    Rm354  Research & Development         Research Scientist   
3    Rm659  Research & Development         Research Scientist   
4    Rm315  Research & Development                    Manager   
5   Rm1279  Research & Development  Healthcare Representative   
6    Rm579  Research & Development     Manufacturing Director   
7    Rm452  Research & Development     Manufacturing Director   
8    Rm271                    None                    Manager   
9    Rm883  Research & Development     Manufacturing Director   
10   Rm890  Research & Development         Research Scientist   
11   Rm613                   Sales            Sales Executive   
12  Rm1304  Research & Development     Manufacturing Director   
13   Rm264                   Sales                    Manager   
14   Rm887  Research & De

In [6]:
#-- Rank JobRoles within each Department by attrition rate
query = """
SELECT Department, JobRole, attrition_rate_pct,
       RANK() OVER (PARTITION BY Department ORDER BY attrition_rate_pct DESC) AS rank_within_dept
FROM (
    SELECT Department, JobRole,
           ROUND(100.0 * SUM(CASE WHEN Attrition='Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
    FROM HR_analytics
    GROUP BY Department, JobRole
);
"""
result = pd.read_sql(query, engine)
print(result)

                Department                    JobRole  attrition_rate_pct  \
0                     None      Laboratory Technician                25.0   
1                     None         Research Scientist                 9.1   
2                     None  Healthcare Representative                 0.0   
3                     None            Human Resources                 0.0   
4                     None                    Manager                 0.0   
5                     None     Manufacturing Director                 0.0   
6                     None          Research Director                 0.0   
7                     None            Sales Executive                 0.0   
8          Human Resources            Human Resources                25.6   
9          Human Resources                    Manager                 0.0   
10  Research & Development      Laboratory Technician                23.8   
11  Research & Development         Research Scientist                17.6   